# Toxic Content Detection — Full Training on GPU (Google Colab)

This notebook runs the **full** training pipeline on a free GPU, for real
(publishable) results — unlike the quick CPU smoke run in the repo.

**Before you start:** set the runtime to GPU —
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save`.

Then run every cell top to bottom (`Runtime -> Run all`). The whole thing
takes roughly 5–15 minutes on a T4.


## 1. Confirm the GPU is active


In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to GPU!')

## 2. Clone the repo and install deps

Colab already ships a GPU build of PyTorch, so we install only the extra
libraries (reinstalling torch could break the CUDA build).


In [ ]:
!git clone https://github.com/SocoSec/ToxicContentDetection.git
%cd ToxicContentDetection
!pip install -q "transformers>=4.46,<5" "scikit-learn>=1.5" "seaborn>=0.13"

## 3. Download the datasets

**Davidson (public, no login)** is fetched automatically below.

The other two are optional but recommended for a real cyberbullying class —
see the optional cell after this one.


In [ ]:
import urllib.request, ssl
from pathlib import Path
Path('data/raw').mkdir(parents=True, exist_ok=True)
url = 'https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv'
req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
data = urllib.request.urlopen(req, timeout=60, context=ssl.create_default_context()).read()
Path('data/raw/labeled_data.csv').write_bytes(data)
print(f'Davidson downloaded: {len(data):,} bytes')

### (Optional) Add the Kaggle Cyberbullying + HatEval datasets

For a **real** cyberbullying class (the repo otherwise tops it up with synthetic
rows), add these. Kaggle needs a free API token:

1. kaggle.com -> your profile -> Settings -> API -> *Create New Token* -> downloads `kaggle.json`.
2. Run the cell, then upload `kaggle.json` when prompted.

Leave this cell unrun to train on Davidson + synthetic top-up only.


In [ ]:
# --- OPTIONAL: uncomment to add the Kaggle cyberbullying dataset ---
# from google.colab import files
# print('Upload your kaggle.json:'); files.upload()
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d andrewmvd/cyberbullying-classification -p data/raw --unzip
# # HatEval (SemEval-2019 Task 5) is gated; if you have the CSV, upload it as:
# #   data/raw/hateval2019_en_train.csv
print('Skipped — training on Davidson + synthetic cyberbullying top-up.')

## 4. Build the dataset and split it


In [ ]:
!python src/dataset_loader.py
!python src/preprocess.py

## 5. Train (full run)

Full defaults: all ~17k training rows, 3 epochs, max_len 128. The scripts
auto-detect the GPU, so no flags are needed. A larger batch size speeds it up
on a T4.


In [ ]:
import os
os.environ['TOXIC_BATCH_SIZE'] = '32'   # T4 handles 32 comfortably at max_len 128
!python src/train.py

## 6. Evaluate on the unseen test set


In [ ]:
!python evaluation/evaluate.py
import json
print(json.dumps(json.load(open('evaluation/metrics.json')), indent=2))

### Confusion matrix


In [ ]:
from IPython.display import Image, display
display(Image('evaluation/confusion_matrix.png'))

## 7. Adversarial robustness test


In [ ]:
!python adversarial/adversarial_test.py
import json
print(json.dumps(json.load(open('adversarial/adversarial_results.json')), indent=2))

## 8. Save the trained model

Download the fine-tuned model (weights + tokenizer) as a zip, or copy it to
your Google Drive for reuse.


In [ ]:
import shutil
shutil.make_archive('toxic_roberta_trained', 'zip', 'models/toxic_roberta')
print('Zipped -> toxic_roberta_trained.zip')
try:
    from google.colab import files
    files.download('toxic_roberta_trained.zip')
except Exception as e:
    print('Download via the Files panel on the left. (', e, ')')
# To save to Drive instead:
#   from google.colab import drive; drive.mount('/content/drive')
#   shutil.copytree('models/toxic_roberta', '/content/drive/MyDrive/toxic_roberta')

---
Trained the model? Compare the macro-F1 here against the repo's CPU baseline
(0.45) — the full GPU run on real data should be substantially higher. Those
are the numbers to put in the LinkedIn post and the README.
